# Eymann_Methodentest_Korpus2.ipynb

AP7 Punkt 1–2: Machbarkeitscheck BERTopic auf einer Teilstichprobe von Korpus 2
(Rechtsdiskurs). Nutzt die per Swisslex gesammelten Aufsätze in `daten/korpus2/pdf/`
und die Metadaten in `Eymann_Korpus2_metadaten.csv`.

**Ziel:** Reicht diese Teilstichprobe methodisch, um mit BERTopic sinnvolle,
interpretierbare Topics zu erhalten? Kein finaler AP9-Lauf, sondern ein
Feasibility-Check gemäss `Workflow_Arbeitspakete.md` AP7.

**Hinweis:** Dieses Notebook braucht Internetzugang (Download des Embedding-Modells)
und `bertopic`/`sentence-transformers` — lokal ausführen, nicht im Cowork-Sandbox.

## Schritt 1 — Metadaten laden

In [4]:
import pandas as pd
import re
from pathlib import Path

PFAD_METADATEN = "../daten/korpus2/Eymann_Korpus2_metadaten.csv"
PFAD_PDF_ORDNER = Path("../daten/korpus2/pdf")

df_meta = pd.read_csv(PFAD_METADATEN)
print(f"{len(df_meta)} Dokumente in der Metadatentabelle")
df_meta.head()


39 Dokumente in der Metadatentabelle


,id,titel_datei,quelle,quelle_typ,jahr,datum_exakt,autor,pre_post,pfad
0,1,"AJP 2022 S. 984, C. Reiter.pdf",AJP,fachartikel,2022.0,NaN,C. Reiter,"PRUEFEN (2022, Monat noetig)","daten/korpus2/pdf/AJP 2022 S. 984, C. Reiter.pdf"
1,2,"AJP 2024 S. 1082, C. Lötscher et al..pdf",AJP,fachartikel,2024.0,NaN,C. Lötscher et al.,post,"daten/korpus2/pdf/AJP 2024 S. 1082, C. Lötsche..."
2,3,"AJP 2024 S. 1306, C. Müller.pdf",AJP,fachartikel,2024.0,NaN,C. Müller,post,"daten/korpus2/pdf/AJP 2024 S. 1306, C. Müller.pdf"
3,4,"AJP 2024 S. 195, C. Yacoubian.pdf",AJP,fachartikel,2024.0,NaN,C. Yacoubian,post,"daten/korpus2/pdf/AJP 2024 S. 195, C. Yacoubia..."
4,5,"AJP 2025 S. 598, I. Wildhaber et al..pdf",AJP,fachartikel,2025.0,NaN,I. Wildhaber et al.,post,"daten/korpus2/pdf/AJP 2025 S. 598, I. Wildhabe..."


## Schritt 2 — Dokumente mit fehlgeschlagener Textextraktion ausschliessen

Zwei Dateien liefern beim Extrahieren keinen brauchbaren Text (geprüft ausserhalb
dieses Notebooks): `Schimpel_KI.pdf` (ZORA-Repository, vermutlich Scan ab Seite 2)
und `inside-it_...pdf` (Print-to-PDF eines Webartikels, kein Textlayer). Für den
PoC werden sie ausgeschlossen; für die spätere volle Erhebung (AP6) brauchen sie
OCR oder eine manuelle Nacherhebung.

In [ ]:
AUSGESCHLOSSEN_POC = ["Schimpel_KI.pdf"]
AUSGESCHLOSSEN_POC += [f for f in df_meta["titel_datei"] if f.lower().startswith("inside-it")]

df_poc = df_meta[~df_meta["titel_datei"].isin(AUSGESCHLOSSEN_POC)].reset_index(drop=True)
print(f"{len(df_poc)} Dokumente fuer den PoC (von {len(df_meta)}), ausgeschlossen: {AUSGESCHLOSSEN_POC}")


## Schritt 3 — Volltextextraktion

In [ ]:
import pdfplumber

def text_extrahieren(pfad):
    """Extrahiert Volltext aus einem PDF. Gibt leeren String zurueck bei Fehler."""
    try:
        with pdfplumber.open(pfad) as pdf:
            seiten = [p.extract_text() or "" for p in pdf.pages]
        return "\n\n".join(seiten)
    except Exception as e:
        print(f"FEHLER bei {pfad}: {e}")
        return ""

texte = {}
for _, row in df_poc.iterrows():
    pfad = Path("..") / row["pfad"]
    texte[row["titel_datei"]] = text_extrahieren(pfad)

woerter = {k: len(v.split()) for k, v in texte.items()}
print("Wortanzahl pro Dokument (Volltext):")
for k, v in woerter.items():
    marker = "  <-- kurz, pruefen" if v < 200 else ""
    print(f"  {k[:55]:55s} {v}{marker}")


## Schritt 4 — Grobe Bereinigung & Segmentierung

Für den PoC reicht eine leichte Bereinigung (keine vollständige AP6-Pipeline):
Kopf-/Fusszeilen-Wiederholungen und offensichtliches Rauschen entfernen, dann in
Absätze segmentieren. BERTopic braucht viele kurze Einheiten statt weniger langer
Dokumente — Segmentierung auf Absatzebene liefert dafür deutlich mehr Datenpunkte
als 37 ganze Artikel.

In [ ]:
def grob_bereinigen(text):
    # Swisslex-Druckkopfzeile auf jeder Seite, z.B. "Ausdruckseite 2 von 19"
    text = re.sub(r"Ausdruckseite\s+\d+\s+von\s+\d+", " ", text)
    # URLs/DOIs aus Fussnoten (z.B. https://doi.org/...) -- kein Inhalt, aber
    # taucht sonst als eigenes "Wort" (https) im Topic-Vokabular auf.
    text = re.sub(r"https?://\S+", " ", text)
    # Häufige Seitenzahl-/Kopfzeilenmuster entfernen (heuristisch, kein Ersatz fuer AP6)
    text = re.sub(r"\n\s*\d{1,4}\s*\n", "\n", text)  # isolierte Seitenzahlen
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

def ist_swisslex_zitierseite(absatz):
    """Erkennt die von Swisslex angehaengte Zitier-/Metadatenseite pro Dokument
    (User-ID, Verlag/Herausgeber/Autor/Titel/Seiten) -- kein Inhalt, muss raus."""
    marker = ["user-id:", "verlag", "herausgeber"]
    tief = absatz.lower()
    return sum(m in tief for m in marker) >= 2

def in_absaetze_segmentieren(text, mindestwoerter=20):
    text = grob_bereinigen(text)
    rohabsaetze = re.split(r"\n\s*\n", text)
    absaetze = [a.strip().replace("\n", " ") for a in rohabsaetze]
    absaetze = [a for a in absaetze if len(a.split()) >= mindestwoerter]
    absaetze = [a for a in absaetze if not ist_swisslex_zitierseite(a)]
    return absaetze

segmente = []
for _, row in df_poc.iterrows():
    for absatz in in_absaetze_segmentieren(texte[row["titel_datei"]]):
        segmente.append({
            "titel_datei": row["titel_datei"],
            "quelle": row["quelle"],
            "quelle_typ": row["quelle_typ"],
            "jahr": row["jahr"],
            "pre_post": row["pre_post"],
            "text": absatz,
        })

df_segmente = pd.DataFrame(segmente)
print(f"{len(df_segmente)} Absatz-Segmente aus {df_poc.shape[0]} Dokumenten")
df_segmente["quelle"].value_counts()


## Schritt 5 — Deskriptive Kontrolle vor dem Topic-Modell

Kurzer Blick auf die Grundgesamtheit, bevor BERTopic läuft: reicht die Menge an
Segmenten überhaupt für einen aussagekräftigen Test? Faustregel aus dem Workflow:
BERTopic braucht typischerweise mehrere hundert Dokumente/Einheiten – hier sind es
Absätze aus 37 Artikeln, deutlich kleiner als der spätere Vollkorpus.

In [ ]:
print("Segmente pro pre/post:")
print(df_segmente["pre_post"].value_counts())
print()
print("Segmente pro Quelle:")
print(df_segmente["quelle"].value_counts())
print()
print("Wortanzahl pro Segment (Verteilung):")
print(df_segmente["text"].str.split().str.len().describe())


## Schritt 6 — BERTopic-Testlauf

Mehrsprachiges Embedding-Modell (Deutsch/Französisch-Anteil im Korpus erwartet,
ähnliche Überlegung wie beim Zero-Shot-Modell für Korpus 1). Braucht Internet für
den Modell-Download beim ersten Lauf.

**Wichtig:** BERTopics Topic-Wörter kommen aus c-TF-IDF über einen `CountVectorizer`,
der standardmässig nur englische Stopwords kennt. Bei einem deutsch/französisch-
dominierten Korpus dominieren sonst Funktionswörter (der/die/und/von ...) die
Topic-Beschreibung. Deshalb eine eigene, mehrsprachige Stopwortliste + zusätzlich
die corpus-definierenden Begriffe (ki/ai/künstliche/intelligenz/artificial/
intelligence), die in praktisch jedem Segment vorkommen und daher nicht
trennscharf sind.

Ausserdem ein fixer `random_state` für UMAP: ohne diesen liefert jeder Lauf
eine andere Anzahl/Zusammensetzung von Topics rein durch die stochastische
Dimensionsreduktion — nicht reproduzierbar, unabhängig von allen anderen
Fixes.

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from umap import UMAP

STOPWORDS_DE = {
    "der","die","das","und","oder","in","im","von","vom","zu","zum","zur","auf","für",
    "mit","dem","den","des","ein","eine","einer","eines","einem","einen","ist","sind",
    "war","waren","wird","werden","wurde","wurden","sich","auch","als","an","am","bei",
    "durch","nicht","nur","noch","schon","so","dass","dass","aber","wenn","wie","wo",
    "wobei","was","wer","wem","wen","welche","welcher","welches","kann","können","muss",
    "müssen","soll","sollen","darf","dürfen","hat","haben","hatte","hatten","sein","ihr",
    "ihre","ihres","ihrer","ihren","ihm","ihn","es","er","sie","wir","man","aus","über",
    "unter","zwischen","vor","nach","bis","um","ohne","gegen","seit","während","wegen",
    "diese","dieser","dieses","diesem","diesen","jene","jener","jenes","dabei","daher",
    "damit","dadurch","davon","dazu","also","denn","doch","etwa","ff","z.b","bzw","vgl",
    "dies","ob","sowie","sowohl","hierzu","hierbei","jeweils","insbesondere","zudem",
    "zwar","sodass","weder","noch","je","desto","ferner","zunächst","zumal","statt",
}
STOPWORDS_FR = {
    "le","la","les","de","des","du","un","une","et","ou","en","dans","pour","sur","avec",
    "par","au","aux","ce","ces","cette","est","sont","ont","être","avoir","que","qui",
    "quoi","dont","où","ne","pas","plus","comme","mais","donc","son","sa","ses","leur",
    "leurs","il","elle","ils","elles","nous","vous","on",
}
STOPWORDS_DOMAENE = {
    "ki","ai","künstliche","künstlichen","intelligenz","artificial","intelligence",
}
STOPWORDS_ZITATION = {
    # Juristischer Zitierapparat (Fussnoten, Gesetzesverweise, Querverweise) --
    # in Fachartikeln extrem haeufig, aber kein inhaltliches Signal.
    "fn","art","ff","siehe","seqq","abs","lit","bge","ziff","rz","erw",
    "vgl","zit","a.a.o","ebd","hrsg","sog",
}
# Blanke Jahreszahlen aus Zitationen (z.B. "AJP 2024", "BGE 2023") als eigene
# Stopwortliste statt per token_pattern -- ein eigener token_pattern hat beim
# BERTopic-internen c-TF-IDF-Schritt (Vokabular ueber die pro Topic
# zusammengefassten Texte) zu "empty vocabulary"-Fehlern gefuehrt, vermutlich
# im Zusammenspiel mit min_df bei sehr wenigen Topic-Gruppen. Eine explizite
# Liste ist hier robuster.
STOPWORDS_JAHRE = {str(jahr) for jahr in range(1990, 2027)}
STOPWORDS_KOMBINIERT = (
    list(ENGLISH_STOP_WORDS) + list(STOPWORDS_DE) + list(STOPWORDS_FR)
    + list(STOPWORDS_DOMAENE) + list(STOPWORDS_ZITATION) + list(STOPWORDS_JAHRE)
)

vectorizer_modell = CountVectorizer(
    stop_words=STOPWORDS_KOMBINIERT,
    ngram_range=(1, 2),
    min_df=2,
)

# BERTopics Standard-UMAP hat keinen fixen random_state -- ohne das liefert
# jeder Lauf eine andere Anzahl/Zusammensetzung von Topics (rein durch die
# stochastische Dimensionsreduktion), unabhaengig von Bereinigung/Vectorizer.
# Fuer Reproduzierbarkeit (und damit Laeufe vergleichbar bleiben) hier fix.
umap_modell = UMAP(random_state=42, n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine")

embedding_modell = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
topic_modell = BERTopic(
    embedding_model=embedding_modell,
    vectorizer_model=vectorizer_modell,
    umap_model=umap_modell,
    language="multilingual",
    calculate_probabilities=False,
)

topics, probs = topic_modell.fit_transform(df_segmente["text"].tolist())
df_segmente["topic"] = topics

print(f"{topic_modell.get_topic_info().shape[0] - 1} Topics gefunden (ohne Outlier-Topic -1)")
topic_modell.get_topic_info().head(15)


## Schritt 7 — Topics inhaltlich prüfen

Für jedes Topic die Top-Wörter und 2–3 Beispielsegmente ansehen: Ergeben die
Topics inhaltlich Sinn (z.B. "Haftung", "Verfahrensrecht", "Datenschutz"), oder
sind sie diffus/beliebig? Das ist der eigentliche Feasibility-Check.

In [ ]:
for topic_id in topic_modell.get_topic_info()["Topic"].head(10):
    if topic_id == -1:
        continue
    woerter = [w for w, _ in topic_modell.get_topic(topic_id)[:8]]
    beispiele = df_segmente[df_segmente["topic"] == topic_id]["text"].head(2).tolist()
    print(f"--- Topic {topic_id}: {', '.join(woerter)} ---")
    for b in beispiele:
        print("  ", b[:150], "...")
    print()


## Schritt 8 — Feasibility-Einschätzung festhalten (AP7 Punkt 4/6)

Manuell nach Durchsicht der Topics oben ausfüllen:

- Ergeben die Topics inhaltlich Sinn? (ja/teilweise/nein)
- Wie viele Segmente landen im Outlier-Topic (-1)? Ist das ein grosser Anteil?
- Reicht die Sample-Grösse (Absätze aus 37 Artikeln), oder braucht es für ein
  belastbares Ergebnis deutlich mehr Dokumente vor dem echten AP9-Lauf?
- Vorläufiger Entscheid: BERTopic weiterverfolgen, mit welchen Anpassungen
  (z.B. `min_topic_size`, mehr Daten, anderes Embedding-Modell)?

In [ ]:
anteil_outlier = (df_segmente["topic"] == -1).mean()
print(f"Anteil Outlier-Segmente (Topic -1): {anteil_outlier:.1%}")


## Schritt 9 — Kodiertabelle für die manuelle Boundary-Work-Kodierung

Exportiert die repräsentativen Segmente pro Topic (via `get_representative_docs()`)
zusammen mit den Metadaten (Quelle, Jahr, pre/post) in eine Tabelle, die direkt für
die manuelle Kodierung nach `Eymann_Codierregeln_BoundaryWork.md` genutzt werden kann.
Die Spalten `modus` und `begruendung_zitat` bleiben leer -- von Hand ausfuellen: pro
Segment den passendsten Modus eintragen (Defending/Creating/Negotiating/Coalescing/
neutral) und ein kurzes Stichwort/Zitat als Beleg.

Outlier-Topic (-1) wird ausgeschlossen, da "repraesentative Dokumente" fuer ein
Rausch-Cluster kein sinnvolles Konzept sind.

In [ ]:
zeilen_kodiertabelle = []
for topic_id in topic_modell.get_topic_info()["Topic"]:
    if topic_id == -1:
        continue
    topic_woerter = ", ".join(w for w, _ in topic_modell.get_topic(topic_id)[:6])
    repr_texte = topic_modell.get_representative_docs(topic_id)
    for text in repr_texte:
        treffer = df_segmente[df_segmente["text"] == text]
        if treffer.empty:
            quelle, jahr, pre_post = "", "", ""
        else:
            zeile = treffer.iloc[0]
            quelle, jahr, pre_post = zeile["quelle"], zeile["jahr"], zeile["pre_post"]
        zeilen_kodiertabelle.append({
            "topic": topic_id,
            "topic_woerter": topic_woerter,
            "quelle": quelle,
            "jahr": jahr,
            "pre_post": pre_post,
            "text": text,
            "modus": "",
            "begruendung_zitat": "",
        })

df_kodiertabelle = pd.DataFrame(zeilen_kodiertabelle)
print(f"{len(df_kodiertabelle)} zu kodierende Segmente aus {df_kodiertabelle['topic'].nunique()} Topics")

PFAD_KODIERTABELLE = "../daten/korpus2/Eymann_Kodiertabelle_BoundaryWork.csv"
df_kodiertabelle.to_csv(PFAD_KODIERTABELLE, index=False)
print(f"Gespeichert nach {PFAD_KODIERTABELLE} -- Spalten 'modus' und 'begruendung_zitat' von Hand ausfuellen.")
df_kodiertabelle.head()


## Schritt 10 — Boundary-Work-Hypothesen-Labels (DE/EN/FR)

AP7 Punkt 3+4 für Korpus 2, analog zum Zero-Shot-Testlauf bei Korpus 1 (Beers 6
Dimensionen). Statt Beers Data Imaginary geht es hier um Faulconbridges 4 Modi
von Boundary Work (`Eymann_Codierregeln_BoundaryWork.md`) plus `neutral`.

**Lehre aus Korpus 1 übernommen:** Einzelwörter als Labels sind zu unspezifisch
für Zero-Shot-NLI (führten bei Beer zu einer 2-Dimensionen-Dominanz trotz hoher
Konfidenz). Deshalb von Anfang an kurze, adjektivische Hypothesen-Phrasen statt
Einzelwörter, passend zu einem festen `hypothesis_template` (nicht volle Sätze
als Label — das ergab bei Beer grammatikalisch kaputten Text und brach die
Konfidenz ein).

**Erster Testlauf (27.07.2026) und Korrektur:** Mit der ersten, eher abstrakt
formulierten Label-Version hatte `defending` 0% Recall (alle 11 tatsächlichen
Fälle wurden falsch klassifiziert), während `creating`/`negotiating`/
`reflexiv/kritisch` als Auffangbecken für eigentlich neutrale, rein dogmatische
Segmente dienten (z.B. Wissenszurechnungs-Theorie als "reflexiv/kritisch"
erkannt, weil akademisch-analytischer Stil oberflächlich nach "kritischem
Hinterfragen" klingt, ohne die spezifische Selbstkritik der Profession zu
meinen). Scores waren durchgehend niedrig (0.18-0.5) -- das Modell war sich bei
keiner der 6 Optionen wirklich sicher, mit `multi_label=False` (Softmax-Zwang)
musste es aber immer einen Sieger küren.

**Fix (in Ruecksprache mit Bernhard):** (1) `reflexiv/kritisch` komplett aus dem
Zero-Shot-Label-Set entfernt -- bei nur 1 bekanntem Fall im ganzen Korpus weder
sinnvoll validierbar noch trennscharf zu formulieren, bleibt aber als manuelle/
qualitative Kategorie in den Codierregeln erhalten (analog zur Datenschutz-
Beobachtung bei Korpus 1: dokumentiert, aber kein Zero-Shot-Ziel). (2) Die
verbleibenden 4 Modi-Labels konkreter formuliert (naeher an den tatsaechlich in
der manuellen Kodierung gefundenen Formulierungsmustern, statt abstrakter
Paraphrasen). (3) `multi_label=True` statt `False` (jedes Label unabhaengig
bewertet statt erzwungener Softmax-Sieger) plus eine Konfidenz-Untergrenze, die
bei durchgehend niedriger Konfidenz automatisch `neutral` waehlt (analog Beers
0.3-Schwelle/"unklar"-Mechanismus).

**Zweiter Testlauf:** weiterhin 0% Recall bei Defending/Creating/Negotiating,
teils sehr hochkonfidente Fehlvorhersagen (0.9-1.0) auf klar neutralem,
dogmatischem Text. Der neu eingefuehrte `score_wahres_label`-Diagnosewert
(Schritt 12/13) zeigte im dritten Testlauf ein gemischtes Bild: Defending
bekommt im Schnitt einen brauchbaren Score (0.406, einzelne Faelle 0.48-0.76),
verliert aber fast immer gegen einen noch hoeheren, oft themenfremden Score bei
`coalescing`/`negotiating` -- ein Konkurrenz-/Schwellenwert-Problem, kein reines
Formulierungsproblem. Negotiating dagegen hat selbst einen sehr niedrigen
wahren Score (0.145, nur n=2) -- hier eher ein Formulierungsproblem, aber bei
nur 2 Faellen wenig belastbar.

**Vierter Fix-Versuch (27.07.2026), zwei parallele Aenderungen:**

1. **Kuerzere Labels.** Die bisherigen Phrasen waren mehrgliedrige Nebensaetze
   ("unternehmerisch, weil sie beschreibt, wie Kanzleien durch KI neue Mandate...")
   -- deutlich laenger/komplexer als Beers erfolgreiche kurze Adjektiv-Phrasen
   ("schnell und zeitsparend"). NLI-Modelle tun sich mit langen, mehrteiligen
   Hypothesen nachweislich schwerer. Zurueck zu kurzen, einfachen Kernphrasen.
2. **Satzebene statt Absatzebene.** Korpus 1 (Beer) klassifizierte auf
   Satzebene, Korpus 2 bisher auf ganzen Absaetzen (oft 50-150+ Woerter,
   durchmischt mit Zitierapparat). Ein einzelner Boundary-Work-Satz kann in
   einem sonst dogmatischen Absatz untergehen und das NLI-Signal verduennen.
   Fix: jeden Absatz in Saetze zerlegen (Schritt 11b), jeden Satz einzeln
   klassifizieren, pro Modus den maximalen Score ueber alle Saetze des Segments
   nehmen (= "kommt dieser Modus irgendwo im Absatz explizit vor?").
   **Bekannter Zielkonflikt:** das kann Recall fuer echte Faelle verbessern
   (der eine relevante Satz wird nicht mehr von umliegendem Text verduennt),
   aber auch False Positives bei echten Neutral-Faellen erhoehen (ein einzelner,
   aus dem Kontext gerissener Satz kann oberflaechlich falsch triggern). Das ist
   eine bewusste, dokumentierte Hypothese fuer diesen Testlauf, kein
   garantierter Fix -- das Ergebnis entscheidet.

In [1]:
HYPOTHESEN_VORLAGE_BW = {
    "de": "Dieser Satz positioniert die Rechtsprofession gegenüber KI als {}.",
    "en": "This sentence positions the legal profession towards AI as {}.",
    "fr": "Cette phrase positionne la profession juridique face à l'IA comme {}.",
}

BOUNDARY_WORK_MODI = {
    "defending": {
        "de": "defensiv: KI darf die juristische Verantwortung nicht übernehmen",
        "en": "defensive: AI must not take over legal responsibility",
        "fr": "défensive : l'IA ne doit pas assumer la responsabilité juridique",
    },
    "creating": {
        "de": "unternehmerisch: neue KI-Beratungsangebote der Kanzlei",
        "en": "entrepreneurial: new AI-related advisory services of the firm",
        "fr": "entrepreneuriale : nouveaux services de conseil en IA du cabinet",
    },
    "negotiating": {
        "de": "kollaborativ: Zusammenarbeit mit Legal Engineers oder Data Scientists",
        "en": "collaborative: working together with legal engineers or data scientists",
        "fr": "collaborative : collaboration avec des legal engineers ou data scientists",
    },
    "coalescing": {
        "de": "umstrukturierend: neues Partnerschaftsmodell der Kanzlei durch KI",
        "en": "restructuring: a new law firm partnership model due to AI",
        "fr": "restructurante : un nouveau modèle de partenariat du cabinet lié à l'IA",
    },
    "neutral": {
        "de": "neutral beschreibend, ohne Positionierung",
        "en": "neutral and purely descriptive, without positioning",
        "fr": "neutre et purement descriptive, sans positionnement",
    },
}

def labels_fuer_sprache_bw(sprache):
    sprache = sprache if sprache in ("de", "en", "fr") else "de"
    return {modus: phrasen[sprache] for modus, phrasen in BOUNDARY_WORK_MODI.items()}

def vorlage_fuer_sprache_bw(sprache):
    return HYPOTHESEN_VORLAGE_BW[sprache if sprache in ("de", "en", "fr") else "de"]


## Schritt 11 — Sprache erkennen

Gleiche Methode wie bei Korpus 1 (`Eymann_Bereinigung_Korpus1.ipynb`):
`langdetect` mit Kurztext-Heuristik als Fallback (langdetect ist bei kurzen
juristischen Zitat-lastigen Absätzen unzuverlässiger als bei den Korpus-1-
Sätzen).

In [5]:
from langdetect import detect

def sprache_erkennen_bw(text):
    try:
        erkannt = detect(text)
        return erkannt if erkannt in ("de", "en", "fr") else de_en_fr_heuristik_bw(text)
    except Exception:
        return de_en_fr_heuristik_bw(text)

def de_en_fr_heuristik_bw(text):
    t = f" {text.lower()} "
    de_marker = [" und ", " ist ", " für ", " nicht ", " werden ", " können ", " auch ", " sind ",
                 " wie ", " sie ", " wir ", " ihr ", " kein", " jede", " gemäss ", " art. "]
    fr_marker = [" et ", " est ", " pour ", " vous ", " nous ", " votre ", " dans ", " avec ", " ses ", " que "]
    en_marker = [" the ", " and ", " for ", " with ", " your ", " you ", " our ", " from ", " this ",
                 " no ", " by ", " shall ", " may "]
    de_score = sum(t.count(m) for m in de_marker) + sum(text.count(c) for c in "äöüßÄÖÜ")
    fr_score = sum(t.count(m) for m in fr_marker)
    en_score = sum(t.count(m) for m in en_marker)
    scores = {"de": de_score, "fr": fr_score, "en": en_score}
    bestimmt = max(scores, key=scores.get)
    return bestimmt if scores[bestimmt] > 0 else "de"


## Schritt 11b — Absätze in Sätze zerlegen (vierter Fix-Versuch)

Einfacher, regelbasierter Satz-Splitter statt einer NLP-Bibliothek (kein
zusätzlicher Download nötig, ausreichend für diesen PoC-Test). Berücksichtigt
die häufigsten juristischen Abkürzungen (Art., Abs., BGE, vgl., ff., z.B. ...),
bei denen ein Punkt keine echte Satzgrenze ist -- sonst würde z.B. "Art. 3 Abs.
2 OR" fälschlich in drei "Sätze" zerrissen. Sehr kurze Fragmente (< 4 Wörter,
meist Abkürzungsreste) werden an den vorherigen Satz angehängt statt als
eigene, bedeutungsarme NLI-Premise verwendet zu werden.

In [6]:
ABKUERZUNGEN_BW = {
    "art", "abs", "ziff", "bge", "vgl", "nr", "lit", "e", "rz", "ff", "bzw", "z.b",
    "d.h", "sog", "hrsg", "a.a.o", "fn", "erw", "seqq", "zit", "s", "co", "resp",
}

def in_saetze_zerlegen_bw(text):
    """Zerlegt einen Absatz in Saetze, ohne bei juristischen Abkuerzungen
    (Art. 3 Abs. 2, BGE 145 III 72, vgl. ...) faelschlich zu trennen."""
    roh = re.split(r'(?<=[.!?])\s+(?=[A-ZÄÖÜ])', text)
    saetze, puffer = [], ""
    for stueck in roh:
        puffer = f"{puffer} {stueck}".strip() if puffer else stueck
        worte = puffer.split()
        letztes_wort = re.sub(r"[.!?]+$", "", worte[-1]).lower() if worte else ""
        if letztes_wort in ABKUERZUNGEN_BW:
            continue  # Punkt ist wahrscheinlich keine Satzgrenze -- weiterlesen
        saetze.append(puffer)
        puffer = ""
    if puffer:
        saetze.append(puffer)

    zusammengefuehrt = []
    for satz in saetze:
        if zusammengefuehrt and len(satz.split()) < 4:
            zusammengefuehrt[-1] = f"{zusammengefuehrt[-1]} {satz}"
        else:
            zusammengefuehrt.append(satz)
    return zusammengefuehrt

# Kurzer Test an einem Beispiel mit Zitierapparat:
beispiel_test = "Die Sorgfaltspflicht ergibt sich aus Art. 398 Abs. 2 OR. KI-Systeme entbinden die Anwältin nicht von dieser Pflicht. Vgl. BGE 145 III 72."
print(in_saetze_zerlegen_bw(beispiel_test))


['Die Sorgfaltspflicht ergibt sich aus Art. 398 Abs. 2 OR.', 'KI-Systeme entbinden die Anwältin nicht von dieser Pflicht.', 'Vgl. BGE 145 III 72.']


## Schritt 12 — Zero-Shot-Klassifikation gegen Bernhards manuelle Kodierung validieren

**Wichtig, anders als bei Beer/Korpus 1:** Hier keine neue Stichprobe ziehen,
sondern direkt gegen die bereits von Bernhard manuell kodierten Segmente
validieren (`Eymann_Kodiertabelle_BoundaryWork.csv`, 42 Segmente aus den
BERTopic-Repräsentanten; `Eymann_Kandidaten_Keywordsuche_BoundaryWork.csv`,
54 gezielt gesuchte Segmente) — das ist der eigentliche Goldstandard-Vergleich
für AP7 Punkt 4 bei Korpus 2.

**Bekannte Limitationen vor dem Lauf:** In keiner der beiden Stichproben kommt
`coalescing` vor (s. Notizen 27.07.2026) — für diesen Modus ist daher weder
Präzision noch Recall aus dem eigenen Korpus prüfbar. Auch für `creating`
(3 Fälle) und `negotiating` (2 Fälle) ist die Fallzahl sehr klein; belastbare
Aussagen sind nur für `neutral` (~78 Fälle) und `defending` (11 Fälle) zu
erwarten. Der eine `reflexiv/kritisch`-Fall wird aus der Validierung
ausgeschlossen, da dieses Label nicht mehr im Zero-Shot-Set ist (s. Schritt 10)
— er kann per Konstruktion nicht mehr korrekt vorhergesagt werden.

**Fix gegenüber dem ersten Testlauf:** `multi_label=True` (jedes Label
unabhängig bewertet, kein erzwungener Softmax-Sieger unter 5 schwachen
Optionen) plus eine Konfidenz-Untergrenze: liegt der höchste Score unter
`KONFIDENZ_UNTERGRENZE_BW`, gilt das Segment automatisch als `neutral`
(analog zu Beers "unklar"-Mechanismus, hier aber direkt auf `neutral`
gemappt, da `neutral` ohnehin die weit überwiegende Klasse ist).

**Vierter Fix-Versuch (nach dem dritten Testlauf, s. Schritt 10):** jedes
Segment wird jetzt in Sätze zerlegt (Schritt 11b), jeder Satz einzeln gegen
alle 5 Labels klassifiziert, und pro Modus der maximale Score über alle Sätze
des Segments übernommen -- ein Segment gilt als "defending", sobald
*irgendein* Satz darin stark dafürspricht, statt dass ein einzelner
relevanter Satz im Absatz-Durchschnitt untergeht. Das ist eine Hypothese, kein
sicherer Fix (s. Diskussion in Schritt 10) -- könnte Recall verbessern, aber
auch False Positives bei Neutral erhöhen.

In [7]:
from transformers import pipeline

klassifikator_bw = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

df_kodiert_1 = pd.read_csv("../daten/korpus2/Eymann_Kodiertabelle_BoundaryWork.csv")
df_kodiert_2 = pd.read_csv("../daten/korpus2/Eymann_Kandidaten_Keywordsuche_BoundaryWork.csv")
df_validierung = pd.concat([
    df_kodiert_1[["quelle", "jahr", "text", "modus"]],
    df_kodiert_2[["quelle", "jahr", "text", "modus"]],
], ignore_index=True)
df_validierung["modus"] = df_validierung["modus"].fillna("").str.strip().str.lower()
df_validierung = df_validierung[df_validierung["modus"] != ""].reset_index(drop=True)

n_reflexiv = (df_validierung["modus"] == "reflexiv/kritisch").sum()
df_validierung = df_validierung[df_validierung["modus"] != "reflexiv/kritisch"].reset_index(drop=True)
print(f"{n_reflexiv} 'reflexiv/kritisch'-Fall/Fälle aus der Validierung ausgeschlossen (Label nicht mehr im Zero-Shot-Set)")
print(f"{len(df_validierung)} manuell kodierte Segmente zur Validierung geladen")
print(df_validierung["modus"].value_counts())

KONFIDENZ_UNTERGRENZE_BW = 0.5

ergebnisse_bw = []
for i, row in df_validierung.iterrows():
    if i % 10 == 0:
        print(f"  ... Segment {i+1}/{len(df_validierung)}")
    sprache = sprache_erkennen_bw(row["text"])
    labels_dict = labels_fuer_sprache_bw(sprache)
    label_zu_modus = {phrase: modus for modus, phrase in labels_dict.items()}
    phrasen = list(labels_dict.values())
    vorlage = vorlage_fuer_sprache_bw(sprache)

    saetze = in_saetze_zerlegen_bw(row["text"]) or [row["text"]]

    # Pro Modus den maximalen Score ueber alle Saetze des Segments -- ein
    # Segment gilt als z.B. "defending", sobald irgendein Satz darin stark
    # dafuerspricht, statt dass ein einzelner relevanter Satz im
    # Absatz-Durchschnitt untergeht (s. Diskussion Schritt 10/vierter Fix).
    score_je_modus_max = {modus: 0.0 for modus in labels_dict}
    for satz in saetze:
        out = klassifikator_bw(satz, candidate_labels=phrasen, multi_label=True, hypothesis_template=vorlage)
        for label, score in zip(out["labels"], out["scores"]):
            modus = label_zu_modus[label]
            if score > score_je_modus_max[modus]:
                score_je_modus_max[modus] = score

    bestes_modus = max(score_je_modus_max, key=score_je_modus_max.get)
    bester_score = score_je_modus_max[bestes_modus]
    vorhersage = "neutral" if bester_score < KONFIDENZ_UNTERGRENZE_BW else bestes_modus

    # Diagnose: Score des TATSAECHLICHEN (manuellen) Labels, unabhaengig davon,
    # ob es "gewinnt" -- unterscheidet zwei Fehlerarten: (a) das wahre Label
    # bekommt selbst einen niedrigen Score (Formulierungsproblem -- das Modell
    # erkennt das Konzept bei echten Faellen gar nicht), oder (b) das wahre
    # Label bekommt einen brauchbaren Score, verliert aber gegen einen noch
    # hoeheren, falschen Score woanders (Konkurrenz-/Schwellenwert-Problem).
    score_wahres_label = score_je_modus_max.get(row["modus"])

    ergebnisse_bw.append({
        "quelle": row["quelle"],
        "jahr": row["jahr"],
        "sprache": sprache,
        "n_saetze": len(saetze),
        "modus_manuell": row["modus"],
        "modus_vorhergesagt": vorhersage,
        "score": round(bester_score, 3),
        "score_wahres_label": round(score_wahres_label, 3) if score_wahres_label is not None else None,
        "text": row["text"][:200],
    })

df_ergebnisse_bw = pd.DataFrame(ergebnisse_bw)
df_ergebnisse_bw["korrekt"] = df_ergebnisse_bw["modus_manuell"] == df_ergebnisse_bw["modus_vorhergesagt"]
print(f"\nGesamtgenauigkeit: {df_ergebnisse_bw['korrekt'].mean():.1%}")


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

1 'reflexiv/kritisch'-Fall/Fälle aus der Validierung ausgeschlossen (Label nicht mehr im Zero-Shot-Set)
95 manuell kodierte Segmente zur Validierung geladen
modus
neutral        79
defending      11
creating        3
negotiating     2
Name: count, dtype: int64
  ... Segment 1/95
  ... Segment 11/95
  ... Segment 21/95
  ... Segment 31/95
  ... Segment 41/95
  ... Segment 51/95
  ... Segment 61/95
  ... Segment 71/95
  ... Segment 81/95
  ... Segment 91/95

Gesamtgenauigkeit: 16.8%


## Schritt 13 — Konfusionsmatrix, Per-Klasse-Auswertung und Diagnose

Wegen der stark ungleichen Klassenverteilung (78 neutral vs. teils nur 1-3
Fälle bei den anderen Modi) ist die Gesamtgenauigkeit allein irreführend — ein
Modell, das immer "neutral" vorhersagt, hätte bereits eine hohe Gesamtgenauigkeit.
Massgeblich ist die Betrachtung pro Klasse (Recall: wie viele der tatsächlichen
Defending/Creating/... Fälle werden auch als solche erkannt?).

**Zusätzliche Diagnose (nach den enttäuschenden ersten beiden Testläufen):**
Der Recall allein sagt nicht, *warum* ein Modus nicht erkannt wird. Deshalb hier
zusätzlich der durchschnittliche Score, den das Modell dem *tatsächlichen*
manuellen Label zuweist -- unabhängig davon, ob dieses Label am Ende "gewinnt".
Zwei sehr unterschiedliche Diagnosen sind möglich:

- **Niedriger Score fürs wahre Label** (z.B. Ø < 0.3 bei Defending-Fällen): Das
  Modell erkennt das Konzept bei echten Fällen gar nicht -- Formulierungsproblem,
  weitere Label-Anpassung könnte helfen.
- **Brauchbarer Score fürs wahre Label, trotzdem falsch vorhergesagt** (z.B.
  Ø > 0.5, aber ein anderes Label bekommt noch höhere Scores): Konkurrenz-
  /Schwellenwert-Problem -- das Modell "sieht" das Richtige durchaus, aber ein
  falsches Label ist noch überzeugender. Das würde eher dafür sprechen, dass die
  *anderen* Labels zu unspezifisch/zu leicht triggerbar sind, nicht dass
  Defending falsch formuliert ist.

In [8]:
konfusion = pd.crosstab(df_ergebnisse_bw["modus_manuell"], df_ergebnisse_bw["modus_vorhergesagt"],
                        rownames=["manuell"], colnames=["vorhergesagt"])
print("Konfusionsmatrix (Zeilen = manuelle Kodierung, Spalten = Modell-Vorhersage):")
print(konfusion)
print()

print("Recall pro manuellem Modus + Diagnose (Score furs wahre Label, korrekt vs. falsch):")
for modus in df_ergebnisse_bw["modus_manuell"].unique():
    teilmenge = df_ergebnisse_bw[df_ergebnisse_bw["modus_manuell"] == modus]
    recall = teilmenge["korrekt"].mean()
    score_wahr_mittel = teilmenge["score_wahres_label"].mean()
    score_wahr_bei_falsch = teilmenge.loc[~teilmenge["korrekt"], "score_wahres_label"].mean()
    print(f"  {modus:20s} n={len(teilmenge):3d}  recall={recall:.1%}  "
          f"durchschn_score_wahres_label={score_wahr_mittel:.3f}  "
          f"durchschn_score_wahres_label_bei_Fehlern={score_wahr_bei_falsch:.3f}")

print()
print("Falsch klassifizierte Faelle (zur manuellen Durchsicht):")
falsch = df_ergebnisse_bw[~df_ergebnisse_bw["korrekt"]]
for _, row in falsch.iterrows():
    print(f"  [{row['quelle']} {row['jahr']}] manuell={row['modus_manuell']} (score_wahres_label={row['score_wahres_label']}) "
          f"-> vorhergesagt={row['modus_vorhergesagt']} (score={row['score']})")
    print(f"    \"{row['text']}...\"")


Konfusionsmatrix (Zeilen = manuelle Kodierung, Spalten = Modell-Vorhersage):
vorhergesagt  coalescing  creating  defending  negotiating  neutral
manuell                                                            
creating               0         3          0            0        0
defending              4         3          2            2        0
negotiating            0         2          0            0        0
neutral               23        18          4           23       11

Recall pro manuellem Modus + Diagnose (Score furs wahre Label, korrekt vs. falsch):
  neutral              n= 79  recall=13.9%  durchschn_score_wahres_label=0.214  durchschn_score_wahres_label_bei_Fehlern=0.116
  defending            n= 11  recall=18.2%  durchschn_score_wahres_label=0.629  durchschn_score_wahres_label_bei_Fehlern=0.555
  creating             n=  3  recall=100.0%  durchschn_score_wahres_label=0.972  durchschn_score_wahres_label_bei_Fehlern=nan
  negotiating          n=  2  recall=0.0%  durchsc

## Schritt 14 — Entscheid festhalten (AP7 Punkt 4/6 für Korpus 2)

Nach Durchsicht der Konfusionsmatrix und der Fehlklassifikationen oben von Hand
ausfüllen:

- Wird `neutral` zuverlässig erkannt (hohe Fallzahl, sollte hohen Recall haben)?
- Wird `defending` (11 Fälle) brauchbar erkannt, oder verwechselt das Modell es
  systematisch mit `neutral` oder `reflexiv/kritisch`?
- Sind die Label-Formulierungen für `creating`/`negotiating` zu vage oder zu
  spezifisch (bei nur 2-3 Fällen wenig belastbar, aber als erster Hinweis
  nutzbar)?
- `coalescing` kann nicht validiert werden (keine Fälle im Korpus) — muss im
  Methodikteil als Limitation offen benannt werden, nicht stillschweigend
  übergangen.
- Vorläufiger Entscheid: Label-Formulierungen anpassen und erneut testen, oder
  Pipeline für den Volltextlauf (analog AP8/AP9 bei Korpus 1) übernehmen?

Entscheid zusätzlich in `Eymann_Notizen_Methodik.md` festhalten.